# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR    = Path('/content/drive/MyDrive/pixtopix')
DRIVE_IMAGES = Path('/content/drive/MyDrive/pixtopix/images/images')

for csv_name in ['train.csv', 'val.csv', 'test.csv']:
    src = DRIVE_DIR / csv_name
    dst = Path('/content') / csv_name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f"Copied  {csv_name}")
    elif dst.exists():
        print(f"Present {csv_name}")
    else:
        print(f"WARNING: {csv_name} not found at {src}")

local_images = Path('/content/images')

if local_images.exists() or local_images.is_symlink():
    if local_images.is_symlink():
        local_images.unlink()
    else:
        shutil.rmtree(local_images)

os.symlink(DRIVE_IMAGES, local_images)
print(f"Symlinked {DRIVE_IMAGES} -> {local_images}")

for split in ['train', 'val', 'test']:
    d = local_images / split
    n = len(list(d.glob('*.png'))) if d.exists() else 0
    print(f"{split:5s} images : {n}")

Mounted at /content/drive
Copied  train.csv
Copied  val.csv
Copied  test.csv
Symlinked /content/drive/MyDrive/pixtopix/images/images -> /content/images
train images : 3109
val   images : 1048
test  images : 1008


In [3]:
import os
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR   = Path("/content")

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 2. Load and Preprocess Data

In [4]:
train_df = pd.read_csv("/content/train.csv")
val_df   = pd.read_csv("/content/val.csv")
test_df  = pd.read_csv("/content/test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [5]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:

    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [6]:
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

CONTENT_DIR = Path("/content")

train_ds = ScienceQADataset(train_df, CONTENT_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   CONTENT_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  CONTENT_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


Datasets created: train=3109, val=1048, test=1008


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [7]:
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    model.to(device)
model.eval()

sample       = val_df.iloc[0]
sample_image = Image.open(Path("/content") / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(text=[sample_prompt], images=[sample_image], return_tensors="pt")
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

## 4. QLoRA Fine-Tuning Setup

We load SmolVLM-500M in **4-bit NF4** quantization and attach LoRA adapters to every
linear projection layer in the language model backbone.
All tunable hyperparameters are collected here — adjust these to climb the leaderboard.

In [8]:
try:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except NameError:
    pass

IMG_SIZE_TRAIN = 384
BATCH_SIZE     = 8
GRAD_ACCUM     = 4
NUM_EPOCHS     = 5
LR             = 2e-4
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.06
MAX_SEQ_LEN    = 3072
LORA_R         = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
SAVE_DIR       = Path('/content/checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

print('Hyperparameters set.')
print(f'  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Image size (train)   : {IMG_SIZE_TRAIN}x{IMG_SIZE_TRAIN}')
print(f'  LoRA rank / alpha    : {LORA_R} / {LORA_ALPHA}')
print(f'  Max sequence length  : {MAX_SEQ_LEN}')
print(f'  Epochs               : {NUM_EPOCHS}')

Hyperparameters set.
  Effective batch size : 32
  Image size (train)   : 384x384
  LoRA rank / alpha    : 8 / 16
  Max sequence length  : 3072
  Epochs               : 5


In [9]:
from peft import LoraConfig, get_peft_model, TaskType

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

model_ft = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)

model_ft.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model_ft.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model_ft = get_peft_model(model_ft, lora_config)
model_ft.print_trainable_parameters()

_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
assert _trainable <= 5_000_000, f'RULE VIOLATION: {_trainable:,} trainable params exceed 5,000,000'
print(f'Competition rule check passed: {_trainable:,} trainable params ≤ 5,000,000')

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Competition rule check passed: 4,784,128 trainable params ≤ 5,000,000


In [10]:
from tqdm.auto import tqdm

train_ds_ft = ScienceQADataset(train_df, Path("/content"), img_size=IMG_SIZE_TRAIN, is_train=True)


def collate_train(batch):

    images = [item['image'] for item in batch]
    texts  = [item['text']  for item in batch]

    enc = processor(
        text=texts,
        images=images,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    labels = enc['input_ids'].clone()
    for i in range(len(batch)):
        labels[i, :] = -100
        non_pad = (enc['input_ids'][i] != processor.tokenizer.pad_token_id).nonzero()
        if len(non_pad):
            last_idx = non_pad[-1].item()
            labels[i, last_idx] = enc['input_ids'][i, last_idx]

    enc['labels'] = labels
    return enc

train_loader = DataLoader(
    train_ds_ft,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
)

print(f'Train batches per epoch: {len(train_loader)}')

Train batches per epoch: 389


In [11]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

num_update_steps = (len(train_loader) * NUM_EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps     = int(num_update_steps * WARMUP_RATIO)

optimizer = AdamW(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, num_update_steps)

scaler = torch.amp.GradScaler('cuda', enabled=False)

model_ft.train()
global_step = 0

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')):
        batch = {k: v.to(model_ft.device) if torch.is_tensor(v) else v for k, v in batch.items()}

        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            outputs = model_ft(**batch)
            loss    = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        running_loss += loss.item() * GRAD_ACCUM

    avg_loss   = running_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | loss: {avg_loss:.4f} | lr: {current_lr:.2e}')

model_ft.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f'Checkpoint saved to: {SAVE_DIR}')

Epoch 1/5:   0%|          | 0/389 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch 1/5 | loss: 0.5734 | lr: 1.89e-04


Epoch 2/5:   0%|          | 0/389 [00:00<?, ?it/s]

Epoch 2/5 | loss: 0.3040 | lr: 1.41e-04


...

Epoch 3/5 | loss: 0.2087 | lr: 7.56e-05


Epoch 4/5:   0%|          | 0/389 [00:00<?, ?it/s]

Epoch 4/5 | loss: 0.1315 | lr: 2.05e-05


Epoch 5/5:   0%|          | 0/389 [00:00<?, ?it/s]

Epoch 5/5 | loss: 0.0972 | lr: 2.12e-08
Checkpoint saved to: /content/checkpoints


In [12]:
import torch.nn.functional as F


@torch.inference_mode()
def predict_log_likelihood(model, processor, dataset, desc='Evaluating'):

    model.eval()
    predictions = []

    for idx in tqdm(range(len(dataset)), desc=desc):
        item      = dataset[idx]
        image     = item['image']
        prompt    = item['text']
        n_choices = len(item['choices'])

        texts = [prompt + f' {CHOICE_LETTERS[i]}' for i in range(n_choices)]

        enc = processor(
            text=texts,
            images=[image] * n_choices,
            return_tensors='pt',
            padding=True,
        )
        enc = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in enc.items()}

        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            logits = model(**enc).logits

        choice_log_probs = []
        for i in range(n_choices):
            letter         = CHOICE_LETTERS[i]
            answer_tok_ids = processor.tokenizer.encode(f' {letter}', add_special_tokens=False)
            n_ans          = len(answer_tok_ids)

            actual_len = int(enc['attention_mask'][i].sum().item())
            log_prob   = 0.0
            for j, tok_id in enumerate(answer_tok_ids):
                pos       = actual_len - n_ans + j - 1
                lp        = F.log_softmax(logits[i, pos], dim=-1)
                log_prob += lp[tok_id].item()

            choice_log_probs.append(log_prob / n_ans)

        predictions.append(int(np.argmax(choice_log_probs)))

    return predictions

In [13]:
val_ds_ft = ScienceQADataset(val_df, Path("/content"), img_size=IMG_SIZE_TRAIN, is_train=False)

val_preds  = predict_log_likelihood(model_ft, processor, val_ds_ft, desc='Validation')
val_labels = val_df['answer'].tolist()
val_acc    = sum(p == l for p, l in zip(val_preds, val_labels)) / len(val_labels)

print(f'Validation Accuracy (log-likelihood): {val_acc:.4f}  ({val_acc * 100:.2f}%)')


Validation:   0%|          | 0/1048 [00:00<?, ?it/s]

Validation Accuracy (log-likelihood): 0.8044  (80.44%)


In [16]:
test_ds_ft = ScienceQADataset(test_df, Path("/content"), img_size=IMG_SIZE_TRAIN, is_train=False)

test_preds = predict_log_likelihood(model_ft, processor, test_ds_ft, desc='Test inference')

submission_df = pd.DataFrame({
    'id':     test_df['id'].values,
    'answer': test_preds,
})
submission_df.to_csv('/content/submission_final.csv', index=False)

print(f'submission_final.csv saved -- {len(submission_df)} predictions')
print(submission_df.head())

Test inference:   0%|          | 0/1008 [00:00<?, ?it/s]

submission_final.csv saved -- 1008 predictions
           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       0
3  test_02425       4
4  test_00930       2
